# util — Photo Export Manifest

Builds a deduplicated list of panorama files to export from the vault for CV training.

**Input**
- `data/processed/leg_photo_candidates.csv` — top-N ranked candidates per leg (notebook 04)

**Output**
- `data/processed/photo_export_manifest.csv` — one row per unique panorama file to export

The same panorama can appear as a candidate for multiple legs (e.g. a photo close to a 4-way
intersection covers multiple approach directions). Deduplication is on `photo_filepath` so each
file appears in the manifest exactly once, regardless of how many legs reference it.

In [ ]:
import os
import pandas as pd

# Base project directory
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"

# Input: top-N candidates per leg produced by notebook 04
CANDIDATES_CSV = os.path.join(PROJECT_DIR, "data", "processed", "leg_photo_candidates.csv")

# Output: deduplicated export manifest
OUTPUT_CSV = os.path.join(PROJECT_DIR, "data", "processed", "photo_export_manifest.csv")

# Which candidate ranks to include in the export.
# 1        = best match per leg only (same photos as leg_photo_selection.csv)
# [1, 2]   = best + one runner-up
# [1,2,3]  = all three ranks (maximum CV training coverage)
# Set to None to include all ranks present in the file.
INCLUDE_RANKS = None  # include all

# Approximate MB per panorama file -- used for storage estimate only.
# Adjust to match the actual export resolution:
#   8000x4000 JPEG ~ 10 MB
#   4000x2000 JPEG ~  4 MB
MB_PER_FILE = 10

print(f"Ranks to include : {INCLUDE_RANKS if INCLUDE_RANKS is not None else 'all'}")
print(f"Assumed file size : {MB_PER_FILE} MB per panorama")

## 1. Load candidates

In [ ]:
# Load the candidate table produced by notebook 04
df = pd.read_csv(CANDIDATES_CSV)
print(f"Candidate rows loaded : {len(df):,}")
print(f"Unique legs           : {df['intersection_id'].astype(str).str.cat(df['leg_bearing'].astype(str), sep='_').nunique():,}")
print(f"Unique intersections  : {df['intersection_id'].nunique():,}")
print(f"Ranks present         : {sorted(df['candidate_rank'].unique())}")
print()

# Show how many legs have each rank available
rank_counts = df.groupby('candidate_rank').size().rename('n_rows')
print("Rows per rank:")
print(rank_counts.to_string())

## 2. Filter ranks and deduplicate

In [ ]:
# Optionally restrict to the desired candidate ranks
if INCLUDE_RANKS is not None:
    filtered = df[df['candidate_rank'].isin(INCLUDE_RANKS)].copy()
    print(f"Rows after rank filter ({INCLUDE_RANKS}): {len(filtered):,}")
else:
    filtered = df.copy()
    print(f"No rank filter applied -- using all {len(filtered):,} rows")

# Deduplicate on photo_filepath: the same panorama can be a candidate for
# multiple legs (e.g. near a 4-way intersection). Keep one row per unique file,
# retaining the lowest candidate_rank (best appearance) for reference.
manifest = (
    filtered
    .sort_values('candidate_rank')          # best rank first
    .drop_duplicates(subset='photo_filepath', keep='first')
    .reset_index(drop=True)
)

print(f"Unique panoramas after deduplication : {len(manifest):,}")
print(f"  (removed {len(filtered) - len(manifest):,} duplicate path references)")

## 3. Summary

In [ ]:
# Storage estimate based on assumed file size
total_gb = len(manifest) * MB_PER_FILE / 1024
print(f"Files to export       : {len(manifest):,}")
print(f"Estimated storage     : {total_gb:.0f} GB  (at {MB_PER_FILE} MB/file)")
print()

# Breakdown by best rank: how many files are exclusively rank-2/3
# (i.e. they would be missed by a rank-1-only export)
rank_breakdown = manifest['candidate_rank'].value_counts().sort_index().rename('n_files')
print("Files by best rank in this export:")
print(rank_breakdown.to_string())
print()

# Preview
manifest[['photo_filepath', 'photo_filename', 'candidate_rank', 'intersection_id']].head(10)

## 4. Save

In [ ]:
# Save the export manifest -- one row per unique panorama file
manifest.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(manifest):,} rows to {OUTPUT_CSV}")